In [3]:
# !pip install opencv-contrib-python pillow pandas numpy
# !pip uninstall opencv-python opencv-contrib-python -y
# !pip install opencv-contrib-python==4.7.0.72


In [1]:
############################################# IMPORTS ################################################
import tkinter as tk
from tkinter import ttk, messagebox as mess, filedialog
import cv2, os, csv, numpy as np
from PIL import Image
import pandas as pd
import datetime
import time
import sqlite3

############################################# HELPERS ################################################
def assure_path_exists(path):
    if not os.path.exists(path):
        os.makedirs(path)

for folder in ["Attendance", "StudentDetails", "TrainingImage", "TrainingImageLabel"]:
    assure_path_exists(folder)

############################################# DATABASE SETUP ##########################################
def create_database():
    conn = sqlite3.connect("AttendanceSystem.db")
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS students (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id TEXT UNIQUE,
        name TEXT
    );
    """)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS attendance (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id TEXT,
        name TEXT,
        date TEXT,
        status TEXT,
        time TEXT
    );
    """)
    conn.commit()
    conn.close()

create_database()

############################################# CAMERA DETECTION ########################################
def get_camera():
    for i in range(3):
        cam = cv2.VideoCapture(i)
        if cam.isOpened():
            cam.release()
            return i
    return -1

camera_index = get_camera()
if camera_index == -1:
    mess.showerror("Camera Error", "No camera detected! Please connect a webcam and restart.")
    exit()

############################################# GUI CLOCK ################################################
def tick():
    time_string = time.strftime('%H:%M:%S')
    clock_label.config(text=time_string)
    clock_label.after(200, tick)

############################################# HAAR FILE CHECK #########################################
def check_haarcascadefile():
    if not os.path.isfile("haarcascade_frontalface_default.xml"):
        mess.showerror("Missing File", "Haarcascade XML file missing! Place it in the same folder.")
        window.destroy()

############################################# CLEAR FIELDS ############################################
def clear():
    txt.delete(0, 'end')
    message_label.config(text="1) Take Images  >>>  2) Train Images")

def clear2():
    txt2.delete(0, 'end')
    message_label.config(text="1) Take Images  >>>  2) Train Images")

############################################# FACE CAPTURE ############################################
def TakeImages():
    global txt, txt2, message_label
    check_haarcascadefile()

    Id = txt.get().strip()
    name = txt2.get().strip()

    if Id == "" or name == "":
        mess.showwarning("Input Error", "Please enter both ID and Name.")
        return
    if not (name.replace(" ", "").isalpha()):
        mess.showwarning("Name Error", "Name must contain only letters and spaces.")
        return

    conn = sqlite3.connect("AttendanceSystem.db")
    cur = conn.cursor()
    cur.execute("SELECT * FROM students WHERE student_id=? OR name=?", (Id, name))
    if cur.fetchone():
        mess.showerror("Duplicate Entry", "Student already registered with this ID or Name.")
        conn.close()
        return
    conn.close()

    cam = cv2.VideoCapture(camera_index)
    cam.set(3, 640)
    cam.set(4, 480)
    detector = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
    sampleNum = 0

    mess.showinfo("Info", "Adjust your face in front of the camera.\nPress 'Q' to quit anytime.")

    while True:
        ret, img = cam.read()
        if not ret:
            mess.showerror("Camera Error", "Cannot access webcam!")
            break
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = detector.detectMultiScale(gray, 1.2, 5, minSize=(100, 100))
        for (x, y, w, h) in faces:
            sampleNum += 1
            cv2.imwrite(f"TrainingImage/{Id}_{name}_{sampleNum}.jpg", gray[y:y+h, x:x+w])
            cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 255), 2)
            cv2.putText(img, f"Samples: {sampleNum}/100", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

        cv2.imshow('Capturing Face - Press Q to stop', img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        elif sampleNum >= 100:
            break

    cam.release()
    cv2.destroyAllWindows()

    conn = sqlite3.connect("AttendanceSystem.db")
    cur = conn.cursor()
    cur.execute("INSERT INTO students (student_id, name) VALUES (?, ?)", (Id, name))
    conn.commit()
    conn.close()

    message_label.config(text=f"Images Taken for ID: {Id}")
    mess.showinfo("Success", f"Images captured successfully for ID: {Id}")

############################################# TRAINING ################################################
def TrainImages():
    check_haarcascadefile()
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    faces, Ids = getImagesAndLabels("TrainingImage")
    if len(faces) == 0:
        mess.showerror("Error", "No images found! Please register someone first.")
        return
    recognizer.train(faces, np.array(Ids))
    recognizer.save("TrainingImageLabel/Trainer.yml")
    message_label.config(text="Training Complete ✔️")
    mess.showinfo("Training Done", "Model has been trained successfully with 100 images per student.")

def getImagesAndLabels(path):
    imagePaths = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.jpg')]
    faces, Ids = [], []
    for imagePath in imagePaths:
        img = Image.open(imagePath).convert('L')
        imageNp = np.array(img, 'uint8')
        Id = int(os.path.split(imagePath)[-1].split('_')[0])
        faces.append(imageNp)
        Ids.append(Id)
    return faces, Ids

############################################# ATTENDANCE ##############################################
def TrackImages():
    global tv
    check_haarcascadefile()
    for k in tv.get_children():
        tv.delete(k)

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.read("TrainingImageLabel/Trainer.yml")
    faceCascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
    cam = cv2.VideoCapture(camera_index)

    conn = sqlite3.connect("AttendanceSystem.db")
    df_students = pd.read_sql_query("SELECT * FROM students", conn)
    conn.close()

    recognized_today = set()
    date = datetime.datetime.now().strftime('%d-%m-%Y')

    while True:
        ret, img = cam.read()
        if not ret:
            mess.showerror("Camera Error", "Cannot access webcam!")
            break
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = faceCascade.detectMultiScale(gray, 1.2, 5)
        for (x, y, w, h) in faces:
            cv2.rectangle(img, (x, y), (x+w, y+h), (0,255,0), 2)
            ID, conf = recognizer.predict(gray[y:y+h, x:x+w])
            if conf < 55:
                matched = df_students[df_students['student_id'].astype(int) == ID]
                if not matched.empty:
                    student_id = str(matched.iloc[0]["student_id"])
                    name = matched.iloc[0]["name"]
                    time_now = datetime.datetime.now().strftime('%H:%M:%S')

                    if student_id not in recognized_today:
                        recognized_today.add(student_id)
                        conn = sqlite3.connect("AttendanceSystem.db")
                        cur = conn.cursor()
                        cur.execute("INSERT INTO attendance (student_id, name, date, status, time) VALUES (?, ?, ?, ?, ?)",
                                    (student_id, name, date, 'Present', time_now))
                        conn.commit()
                        conn.close()
                        tv.insert('', 0, text=student_id, values=(name, date, 'Present', time_now))
                    cv2.putText(img, name, (x, y+h+30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
            else:
                cv2.putText(img, "Unknown", (x, y+h+30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        cv2.imshow('Taking Attendance - Press Q to stop', img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cam.release()
    cv2.destroyAllWindows()

    # Mark absentees
    conn = sqlite3.connect("AttendanceSystem.db")
    for _, row in df_students.iterrows():
        if str(row["student_id"]) not in recognized_today:
            cur = conn.cursor()
            cur.execute("INSERT INTO attendance (student_id, name, date, status, time) VALUES (?, ?, ?, ?, ?)",
                        (row["student_id"], row["name"], date, 'Absent', '-'))
    conn.commit()
    conn.close()

    mess.showinfo("Attendance", "Attendance Recorded Successfully (Present & Absent).")
    export_daily_excel(date)

############################################# EXPORT SEPARATE FILES ###################################
def export_daily_excel(date):
    conn = sqlite3.connect("AttendanceSystem.db")
    df = pd.read_sql_query("SELECT * FROM attendance WHERE date=?", conn, params=(date,))
    conn.close()

    if df.empty:
        mess.showwarning("No Data", "No attendance records found for today.")
        return

    folder = filedialog.askdirectory(title="Select Folder to Save Reports")
    if not folder:
        return

    present_df = df[df["status"] == "Present"]
    absent_df = df[df["status"] == "Absent"]

    present_path = os.path.join(folder, f"Present_{date}.xlsx")
    absent_path = os.path.join(folder, f"Absent_{date}.xlsx")

    present_df.to_excel(present_path, index=False)
    absent_df.to_excel(absent_path, index=False)

    mess.showinfo("Export Complete", f"✅ Reports saved:\n\n{present_path}\n{absent_path}")

############################################# GUI #######################################################
window = tk.Tk()
window.title("AI-Based Classroom Attendance System")
window.geometry("1280x720")
window.configure(bg="#0f172a")

# Header
header_frame = tk.Frame(window, bg="#1e3a8a", height=90)
header_frame.pack(fill="x")
title_label = tk.Label(header_frame, text="🎓 AI-Based Classroom Attendance System",
                       font=('Segoe UI', 26, 'bold'), bg="#1e3a8a", fg="white")
title_label.pack(side="left", padx=40, pady=20)
clock_label = tk.Label(header_frame, font=('Segoe UI', 22, 'bold'), bg="#1e3a8a", fg="#facc15")
clock_label.pack(side="right", padx=40)
tick()

# Left Frame
frame_left = tk.Frame(window, bg="#1e293b", highlightbackground="#334155", highlightthickness=2)
frame_left.place(x=70, y=130, width=540, height=520)
tk.Label(frame_left, text="Attendance Panel", bg="#1e293b", fg="#38bdf8", font=('Segoe UI', 20, 'bold')).pack(pady=15)

tk.Button(frame_left, text="Take Attendance", command=TrackImages, bg="#facc15", fg="black",
          font=('Segoe UI', 12, 'bold'), relief="flat", width=25).pack(pady=10)
tk.Button(frame_left, text="Quit", command=window.destroy, bg="#ef4444", fg="white",
          font=('Segoe UI', 12, 'bold'), relief="flat", width=25).pack(pady=10)

# Attendance Table
tv = ttk.Treeview(frame_left, columns=('name', 'date', 'status', 'time'), height=10)
tv.heading('#0', text='ID')
tv.heading('name', text='Name')
tv.heading('date', text='Date')
tv.heading('status', text='Status')
tv.heading('time', text='Time')
tv.column('#0', width=80)
tv.pack(pady=10, fill="both", expand=True)

# Right Frame
frame_right = tk.Frame(window, bg="#1e293b", highlightbackground="#334155", highlightthickness=2)
frame_right.place(x=680, y=130, width=540, height=520)
tk.Label(frame_right, text="Registration Panel", bg="#1e293b", fg="#38bdf8",
         font=('Segoe UI', 20, 'bold')).pack(pady=15)

tk.Label(frame_right, text="Student ID:", bg="#1e293b", fg="white", font=('Segoe UI', 14)).place(x=60, y=100)
txt = tk.Entry(frame_right, font=('Segoe UI', 14), width=20)
txt.place(x=200, y=100)
tk.Button(frame_right, text="Clear", command=clear, bg="#ef4444", fg="white",
          font=('Segoe UI', 10, 'bold'), relief="flat").place(x=410, y=100)

tk.Label(frame_right, text="Student Name:", bg="#1e293b", fg="white", font=('Segoe UI', 14)).place(x=60, y=160)
txt2 = tk.Entry(frame_right, font=('Segoe UI', 14), width=20)
txt2.place(x=200, y=160)
tk.Button(frame_right, text="Clear", command=clear2, bg="#ef4444", fg="white",
          font=('Segoe UI', 10, 'bold'), relief="flat").place(x=410, y=160)

tk.Button(frame_right, text="Take Images", command=TakeImages, bg="#3b82f6", fg="white",
          font=('Segoe UI', 12, 'bold'), relief="flat", width=20).place(x=150, y=250)
tk.Button(frame_right, text="Train Model", command=TrainImages, bg="#10b981", fg="white",
          font=('Segoe UI', 12, 'bold'), relief="flat", width=20).place(x=150, y=310)

message_label = tk.Label(frame_right, text="", bg="#1e293b", fg="#facc15",
                         font=('Segoe UI', 12, 'italic'))
message_label.place(x=100, y=370)

window.mainloop()


C:\Users\admin\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
